# EXA-STAR: neuro-evolved ViT-MAE foundation model on HCP fMRI (Kaggle T4x2)

Evolves a masked-autoencoder vision transformer whose encoder/decoder internals are an EXAMM-style
block graph, trained on the full subject-organized HCP corpus with a subject-level 70/10/20 split.

**Setup expected:**
- The HCP corpus (subject subdirs of `.npz`) mounted as a **Kaggle Dataset** at `HCP_ROOT`.
- The exa-star repo importable (clone it or mount it as a dataset; see the imports cell).

**Kaggle notes:** sessions time out (~12h) and GPU quota is weekly, so the loop **checkpoints to
`/kaggle/working` and resumes** automatically. This first version trains on a **single T4**; two-GPU
genome-parallelism is a later enhancement (see the final markdown cell).


## 1. Configuration

In [10]:
import os

# --- repo + data locations (EDIT these) ---
REPO_PATH = "/kaggle/working/exa-star"                 # where the repo is cloned on Kaggle
REPO_URL = "https://github.com/axj2613/exa-ae.git"     # your fork; add a token here if private
REPO_BRANCH = "autoencoder-aryan"
HCP_ROOT = "/kaggle/working/hcp_complete"          # dir containing the <subject_id>/ subdirs
ATLAS_COORDS = os.path.join(REPO_PATH, "datasets/hcp/atlases/A424_Coordinates.dat")

WORKDIR = "/kaggle/working"
SPLIT_PATH = os.path.join(WORKDIR, "subject_split.json")     # persisted so resume/eval reuse the SAME split
STATS_PATH = os.path.join(WORKDIR, "norm_stats.npz")         # frozen train-split normalization
LENGTH_INDEX_PATH = os.path.join(WORKDIR, "length_index.json")  # cached recording lengths (header-only scan)
CHECKPOINT_PATH = os.path.join(WORKDIR, "evolution_checkpoint.pkl")
BEST_GENOME_PATH = os.path.join(WORKDIR, "best_genome.pkl")
HISTORY_PATH = os.path.join(WORKDIR, "fitness_history.json")   # best/mean fitness per checkpoint
PROGRESS_PLOT_PATH = os.path.join(WORKDIR, "evolution_progress.png")

# --- windowing ---
# A handful of HCP runs are truncated (some tasks are as short as ~35 timepoints), so no single
# window can include literally every recording without being uselessly tiny. window=120 (6 temporal
# patches) includes 99.7% of recordings -- all complete runs incl. EMOTION -- and the dataset logs
# how many truncated recordings (<window) it excludes. Raise toward 140/160 for more temporal
# context at the cost of dropping a few more short runs.
WINDOW_LENGTH = 120          # must be divisible by TIME_PATCH_SIZE
TIME_PATCH_SIZE = 20         # 120 / 20 = 6 temporal patches per parcel
MASK_RATIO = 0.75
SPLIT_RATIOS = (0.7, 0.1, 0.2)

# --- model / evolution (full-scale search config) ---
D_MODEL = 128                # baked into the topology -- this is your final model width
NUM_HEADS = 4
D_FF = 512                   # 4x d_model, standard FFN size inside each attention block
DROPOUT = 0.1
# this run is the ATTENTION-ONLY arm; set the full list
# ["attention", "simple", "sequence_lstm", "temporal_lstm"] for the mixed-cell comparison.
NODE_TYPES = ["attention"]
POPULATION_SIZE = 16         # larger pool hedges the noisy proxy + gives the re-rank more survivors
NUM_GENERATIONS = 500        # monitor evolution_progress.png; stop early once it plateaus

# --- per-genome training budget (search only; the winner is trained long separately via
#     train_final_model.py) ---
NUM_ITERATIONS = 6            # 6x60 = 360 steps/genome, up from 4x40=160 (see BATCHES_PER_ITERATION)
BATCHES_PER_ITERATION = 60   # more gradient signal per genome + more Lamarckian accumulation/gen, after the last run plateaued at MSE 0.981
BATCH_SIZE = 8               # search batch: small to bound activation memory as genomes grow
FITNESS_BATCHES = 32         # doubled: fitness was over only 128 windows (very noisy ranking, fidelity rho~0); more val batches = less selection noise
LEARNING_RATE = 0.001
USE_AMP = True               # CUDA mixed precision (tensor cores on T4)

CHECKPOINT_EVERY = 5         # genomes between checkpoints


## 2. Fetch the repo + imports

Clones the repo on a fresh session (or pulls the latest on re-run) so the code always matches
this notebook -- no more manual dataset re-uploads. If your fork is **private**, put a GitHub
token in `REPO_URL` (e.g. from Kaggle Secrets): `https://{token}@github.com/axj2613/exa-ae.git`.

*(If you `git pull` new code into an already-imported session, restart the kernel so Python
reloads the modules.)*

In [11]:
import os
import subprocess
import sys

if not os.path.isdir(REPO_PATH):
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull", "origin", REPO_BRANCH], check=True)

if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

import pickle
# expandable segments reduce caching-allocator fragmentation across many differently-sized
# genomes (must be set before torch initializes CUDA -- a fresh kernel picks it up).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")


Already up to date.
device: cuda Tesla T4


From https://github.com/axj2613/exa-ae
 * branch            autoencoder-aryan -> FETCH_HEAD


## 3. Dataset

Builds the subject-level split (persisted) and freezes per-parcel normalization from the training
split (persisted). Both files are reused on resume and by the downstream embedding extraction so
the split and normalization are identical everywhere.

In [12]:
from time_series.hcp_window_dataset import HCPWindowDataset

dataset = HCPWindowDataset(
    root_dir=HCP_ROOT,
    atlas_coordinates_filename=ATLAS_COORDS,
    window_length=WINDOW_LENGTH,
    split_ratios=SPLIT_RATIOS,
    split_path=SPLIT_PATH,
    stats_path=STATS_PATH,
    length_index_path=LENGTH_INDEX_PATH,
)
print("parcels:", dataset.num_parcels)
print("split sizes:", {k: len(v) for k, v in dataset.splits.items()})


2026-07-13 16:48:20.080 | INFO     | time_series.hcp_window_dataset:__init__:110 - discovered 1098 subjects, 19134 recordings under '/kaggle/working/hcp_complete'
2026-07-13 16:48:20.089 | INFO     | time_series.hcp_window_dataset:_build_length_index:187 - loaded recording length index from '/kaggle/working/length_index.json'
2026-07-13 16:48:20.090 | INFO     | time_series.hcp_window_dataset:_make_or_load_split:153 - loaded subject split from '/kaggle/working/subject_split.json'
2026-07-13 16:48:20.105 | WARNING  | time_series.hcp_window_dataset:_index_recordings:227 - excluded 59 recordings shorter than window_length=120 (per split: {'train': 46, 'val': 3, 'test': 10}); 19075 usable recordings remain
2026-07-13 16:48:20.106 | INFO     | time_series.hcp_window_dataset:_load_or_compute_stats:266 - loaded normalization stats from '/kaggle/working/norm_stats.npz'


parcels: 424
split sizes: {'train': 769, 'val': 110, 'test': 219}


## 4. Build or resume the population

If a checkpoint exists (from a previous timed-out session) it is loaded and the run continues;
otherwise a fresh seed genome + population is created.

In [13]:
from population.single_population import SinglePopulation
from evolution.vision_transformer_block_edge_generator import VisionTransformerBlockEdgeGenerator
from evolution.vision_transformer_block_node_generator import VisionTransformerBlockNodeGenerator
from evolution.vision_transformer_block_reproduction_selector import VisionTransformerBlockReproductionSelector
from genomes.vision_transformer_block_genome import VisionTransformerBlockGenome
from weight_generators.lamarckian_block_weight_generator import LamarckianBlockWeightGenerator
from evolution.checkpoint import save_checkpoint, load_checkpoint

if os.path.exists(CHECKPOINT_PATH):
    state = load_checkpoint(CHECKPOINT_PATH)
    population = state["population_strategy"]
    start_generation = state["generation"]
    # the checkpoint was pickled on CPU; re-home the whole population (and seed) to the GPU so
    # it isn't device-mixed with the GPU children generated after resuming.
    for genome in population.population:
        genome.to(device)
    if population.seed_genome is not None:
        population.seed_genome.to(device)
    print(f"resumed from checkpoint at generation {start_generation}")
else:
    weight_generator = LamarckianBlockWeightGenerator()
    node_generator = VisionTransformerBlockNodeGenerator(
        num_heads=NUM_HEADS, d_ff=D_FF, dropout=DROPOUT, allowed_node_types=NODE_TYPES
    )
    edge_generator = VisionTransformerBlockEdgeGenerator()

    seed_genome = VisionTransformerBlockGenome(
        generation_number=0, num_parcels=dataset.num_parcels, window_length=WINDOW_LENGTH,
        parcel_coordinates=dataset.parcel_coordinates, d_model=D_MODEL, num_heads=NUM_HEADS,
        d_ff=D_FF, dropout=DROPOUT, time_patch_size=TIME_PATCH_SIZE, mask_ratio=MASK_RATIO,
        weight_generator=weight_generator,
    )
    population = SinglePopulation(
        population_size=POPULATION_SIZE, seed_genome=seed_genome,
        reproduction_selector=VisionTransformerBlockReproductionSelector(
            node_generator=node_generator, edge_generator=edge_generator, weight_generator=weight_generator,
        ),
    )
    start_generation = 0
    print("starting fresh evolution with node types:", NODE_TYPES)


AcceleratorError: CUDA error: an illegal memory access was encountered
Search for `cudaErrorIllegalAddress' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 5. Evolution loop (multi-GPU)

`resolve_devices()` finds every GPU (both T4s on Kaggle's T4x2), and `evolve_parallel` generates
one genome per device and **trains them concurrently, one per GPU** -- ~2x throughput. Each genome
trains on the **train** split (mixed precision) and is scored on the **validation** split.
Checkpoints every ~`CHECKPOINT_EVERY` genomes survive session restarts.

*(The reproduction operators print verbosely; filter stdout/loguru if you want a quieter log.)*

In [ ]:
from evolution.parallel_training import evolve_parallel, resolve_devices
from evolution.fitness_history import FitnessHistory

config = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, tuple, list))}
devices = resolve_devices()
print("training devices:", devices)
history = FitnessHistory(HISTORY_PATH)   # reloads prior history on resume, so the curve is continuous
train_kwargs = dict(
    iterations=NUM_ITERATIONS, batches_per_iteration=BATCHES_PER_ITERATION,
    batch_size=BATCH_SIZE, fitness_batches=FITNESS_BATCHES, use_amp=USE_AMP,
)

generation = start_generation
last_checkpoint = start_generation
progress = tqdm(total=NUM_GENERATIONS, initial=start_generation)
while generation < NUM_GENERATIONS:
    k = min(len(devices), NUM_GENERATIONS - generation)
    evolve_parallel(population, dataset, devices[:k], LEARNING_RATE, num_genomes=k, **train_kwargs)
    generation += k
    progress.update(k)
    if generation - last_checkpoint >= CHECKPOINT_EVERY:
        # save_checkpoint moves genomes to CPU to pickle, then restores each to its own device.
        save_checkpoint(CHECKPOINT_PATH, population, generation, config=config)
        history.record(generation, population)   # persisted best/mean fitness for the progress plot
        best = population.population[0]
        with open(BEST_GENOME_PATH, "wb") as best_file:
            pickle.dump(best, best_file)
        print(f"[checkpoint @ gen {generation}] best validation MSE: {best.fitness:.6f}")
        last_checkpoint = generation
progress.close()


training devices: [device(type='cuda', index=0), device(type='cuda', index=1)]


 12%|#2        | 60/500 [00:00<?, ?it/s]

REPRODUCITON METHOD: EnableNode
REPRODUCITON METHOD: Crossover
iteration 0 train loss: nan
iteration 0 train loss: 0.994263
iteration 1 train loss: nan
iteration 2 train loss: nan
iteration 1 train loss: 0.981585
iteration 3 train loss: nan
final fitness (validation MSE): nan | active params: 1,448,705 (1,422,285 evolved) | active hidden nodes: 5, edges: 13 | types: {'SimpleBlockNode': 1, 'AttentionBlockNode': 4}
iteration 2 train loss: nan
iteration 3 train loss: nan
final fitness (validation MSE): nan | active params: 2,141,322 (2,114,902 evolved) | active hidden nodes: 7, edges: 22 | types: {'SimpleBlockNode': 1, 'AttentionBlockNode': 6}


POPULATION:
genome[0] generated: 41, fitness: 0.9916225001215935, generated by: DisableEdge
genome[1] generated: 31, fitness: 0.9930920638144016, generated by: AddNode
genome[2] generated: 53, fitness: 0.9945438355207443, generated by: SplitNode
genome[3] generated: 47, fitness: 0.9952206797897816, generated by: DisableEdge
genome[4] generated: 3,

AcceleratorError: CUDA error: an illegal memory access was encountered
Search for `cudaErrorIllegalAddress' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


## 6. Final save + download

All outputs are written to `/kaggle/working` (`WORKDIR`). **`/kaggle/working` is wiped when an
interactive session ends unless you persist it** -- so to keep the trained model do ONE of:
- enable **Persistence -> Files only** in the notebook settings (right sidebar) -- also required
  for the checkpoint/resume across sessions to work; or
- **Save Version** (Save & Run All), which stores `/kaggle/working` as the version's Output; or
- download the files now via the links printed below.

Downstream clinical evaluation (`subject_embeddings.py` -> `eval_cls.py`) needs THREE of these,
not just the genome: `best_genome.pkl`, `subject_split.json`, and `norm_stats.npz` -- so that the
held-out test subjects and the normalization match what the model was trained with.

In [ ]:
from IPython.display import FileLink, display

save_checkpoint(CHECKPOINT_PATH, population, generation, config=config)
history.record(generation, population)
history.plot(PROGRESS_PLOT_PATH)   # best/mean validation MSE vs generation -- watch it plateau
best = population.population[0]
with open(BEST_GENOME_PATH, "wb") as best_file:
    pickle.dump(best, best_file)

print("best genome validation MSE:", best.fitness)
report = best.parameter_report()
print(f"best genome size: {report['total_active_parameters']:,} active trainable params "
      f"({report['evolved_active_parameters']:,} evolved) -- vs BrainLM's 111M / 650M")
print(f"  active hidden blocks: {report['num_active_hidden_nodes']} {report['node_type_counts']}, "
      f"active edges: {report['num_active_edges']}")
print(best)

print("\nartifacts in", WORKDIR, "(download these -- the first three are needed for clinical eval):")
for path in [BEST_GENOME_PATH, SPLIT_PATH, STATS_PATH, CHECKPOINT_PATH, LENGTH_INDEX_PATH,
             HISTORY_PATH, PROGRESS_PLOT_PATH]:
    if os.path.exists(path):
        print(f"  {os.path.getsize(path) / 1e6:8.2f} MB  {path}")
        display(FileLink(os.path.relpath(path, WORKDIR)))


## Notes

- **Multi-GPU** is active: `evolve_parallel` trains one genome per detected GPU concurrently
  (generational batching -- generate K, train across `cuda:0`/`cuda:1`, insert K). It falls back to
  a single device (or CPU) automatically. Set `POPULATION_SIZE` >= number of GPUs.
- **Per-genome budget vs. dataset size:** training cost is fixed by `NUM_ITERATIONS x
  BATCHES_PER_ITERATION x BATCH_SIZE`, not by how much data exists -- each genome sees a small
  random slice. To exploit more data / get a less noisy fitness signal, raise those budgets (and/or
  `NUM_GENERATIONS`). The proxy-vs-full fidelity experiment
  (`evaluation_scripts/proxy_fidelity_experiment.py`) checks whether the cheap fitness ranks
  architectures the way full training would.